# 13 · Vision Transformer: patches, position embeddings, resolution changes

**Needed for:** ViT (vision track, step V1), then every vision-language model: CLIP, SigLIP, LLaVA, Qwen2-VL and
Janus all use a ViT as the image encoder.

A ViT turns an image into a sequence and runs an ordinary transformer encoder on it:
1. Cut the $H\times W$ image into $P\times P$ patches: $N = HW/P^2$ patches.
2. Flatten each patch ($C \cdot P^2$ numbers) and project it linearly to $D$ dimensions.
3. Add position embeddings; optionally prepend a learned `[CLS]` token.
4. Transformer encoder (bidirectional, no causal mask). Classify from `[CLS]` or the mean of all tokens.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"


def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")

## 1. Patchify, two ways

Reshape-based patchify and a `Conv2d` with `kernel_size = stride = P` compute exactly the same thing. The conv
form is what most code uses; the reshape form is what you need when patches have to be moved around (pixel
shuffle in notebook 15, unpatchify in notebook 17). The flatten order must be (channel, row, column) to match
the conv weight layout.

In [ ]:
def patchify(img, p):
    """(B, C, H, W) -> (B, N, C*p*p), patches in row-major order."""
    B, C, H, W = img.shape
    x = img.reshape(B, C, H // p, p, W // p, p)
    x = x.permute(0, 2, 4, 1, 3, 5)                  # (B, H/p, W/p, C, p, p)
    return x.reshape(B, (H // p) * (W // p), C * p * p)


def unpatchify(x, p, C, H, W):
    B = x.shape[0]
    x = x.reshape(B, H // p, W // p, C, p, p).permute(0, 3, 1, 4, 2, 5)
    return x.reshape(B, C, H, W)


def test_patchify(fn):
    img = torch.randn(2, 3, 32, 48)
    x = fn(img, 8)
    assert x.shape == (2, 24, 192)
    torch.testing.assert_close(x[0, 1], img[0, :, 0:8, 8:16].reshape(-1))     # second patch = row 0, col 1
    torch.testing.assert_close(x[0, 6], img[0, :, 8:16, 0:8].reshape(-1))     # 6 patches per row
    conv = nn.Conv2d(3, 64, 8, stride=8, bias=False)
    via_conv = conv(img).flatten(2).transpose(1, 2)                           # (B, N, D)
    torch.testing.assert_close(fn(img, 8) @ conv.weight.reshape(64, -1).T, via_conv, atol=1e-5, rtol=1e-5)


test_patchify(patchify)
img = torch.randn(2, 3, 32, 48)
torch.testing.assert_close(unpatchify(patchify(img, 8), 8, 3, 32, 48), img)
print("patchify == Conv2d(stride=P), and unpatchify inverts it")

## 2. Position embeddings

- **Learned 1-D** (original ViT): a $(N+1)\times D$ table.
- **2-D sin-cos** (MAE, DiT): half the channels encode the row, half the column; no parameters.
- **2-D RoPE** (newer ViTs, Qwen2-VL's encoder): rotate by row and column angles inside attention.

**Changing resolution:** a learned table trained on a $14\times14$ grid has no entry for patch $(20, 20)$.
The standard fix is to reshape the table into its 2-D grid and resize it with bicubic interpolation.

In [ ]:
def sincos_2d(h, w, dim, base=10000.0):
    """(h*w, dim) fixed embedding: first half from row index, second half from column index."""
    assert dim % 4 == 0
    omega = 1.0 / base ** (torch.arange(dim // 4).float() / (dim // 4))
    ys, xs = torch.meshgrid(torch.arange(h).float(), torch.arange(w).float(), indexing="ij")
    y, x = ys.flatten()[:, None] * omega, xs.flatten()[:, None] * omega
    return torch.cat([y.sin(), y.cos(), x.sin(), x.cos()], dim=1)


def interpolate_pos_embed(pos, old_hw, new_hw):
    """pos: (old_h*old_w, D) grid embedding (no CLS). Returns (new_h*new_w, D)."""
    D = pos.shape[-1]
    grid = pos.reshape(1, *old_hw, D).permute(0, 3, 1, 2)              # (1, D, h, w)
    grid = F.interpolate(grid, size=new_hw, mode="bicubic", align_corners=False)
    return grid.permute(0, 2, 3, 1).reshape(new_hw[0] * new_hw[1], D)


def test_interpolate(fn):
    pos = torch.randn(14 * 14, 32)
    assert fn(pos, (14, 14), (20, 24)).shape == (480, 32)
    torch.testing.assert_close(fn(pos, (14, 14), (14, 14)), pos, atol=1e-5, rtol=1e-5)  # same size = no-op
    smooth = sincos_2d(8, 8, 32)
    up = fn(smooth, (8, 8), (16, 16)).reshape(16, 16, 32)
    assert (up[0, 0] - smooth[0]).abs().max() < 0.6   # corner stays close to the original corner


test_interpolate(interpolate_pos_embed)
print("sincos table:", sincos_2d(14, 14, 64).shape, "| resized 14x14 -> 20x24 ok")

## 3. A tiny ViT that actually learns

Synthetic task: images of horizontal vs vertical stripes with random phase, frequency and noise. A single patch
can't always tell them apart, so the model has to combine patches through attention.

In [ ]:
class TinyViT(nn.Module):
    def __init__(self, img=32, patch=8, C=1, dim=64, depth=2, heads=4, n_classes=2):
        super().__init__()
        self.p = patch
        n = (img // patch) ** 2
        self.proj = nn.Linear(C * patch * patch, dim)
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.randn(1, n + 1, dim) * 0.02)
        layer = nn.TransformerEncoderLayer(dim, heads, 4 * dim, dropout=0.0, batch_first=True, norm_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, n_classes)

    def forward(self, img):
        x = self.proj(patchify(img, self.p))
        x = torch.cat([self.cls.expand(x.shape[0], -1, -1), x], 1) + self.pos
        return self.head(self.norm(self.encoder(x))[:, 0])


def stripes(B, size=32):
    label = torch.randint(0, 2, (B,))
    freq = torch.randint(2, 6, (B, 1, 1)).float()
    phase = torch.rand(B, 1, 1) * 2 * math.pi
    coord = torch.arange(size).float()
    rows = coord[None, :, None].expand(B, size, size)
    cols = coord[None, None, :].expand(B, size, size)
    along = torch.where(label[:, None, None] == 0, rows, cols)
    imgs = torch.sin(2 * math.pi * freq * along / size + phase) + 3.0 * torch.randn(B, size, size)
    return imgs[:, None], label


vit = TinyViT().to(device)
opt = torch.optim.AdamW(vit.parameters(), lr=1e-3)
for step in range(201):
    x, y = stripes(64)
    loss = F.cross_entropy(vit(x.to(device)), y.to(device))
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 50 == 0:
        with torch.no_grad():
            xt, yt = stripes(512)
            acc = (vit(xt.to(device)).argmax(-1).cpu() == yt).float().mean().item()
        print(f"step {step:3d}  loss {loss.item():.3f}  test acc {acc:.2f}")

## Things to be able to explain

- Tokens per image: a 336×336 image with 14×14 patches gives 576 tokens (LLaVA-1.5). Why does that make
  high-resolution images expensive in a VLM, and what are the fixes? (Pixel shuffle, pooling, resamplers: notebook 15.)
- Why does ViT need more data than a CNN? (No built-in locality or translation equivariance.)
- `[CLS]` vs mean pooling: both work; SigLIP uses an attention-pooling head instead.

## Drill

In [ ]:
def my_patchify(img, p):
    raise NotImplementedError


def my_interpolate_pos_embed(pos, old_hw, new_hw):
    raise NotImplementedError


drill(test_patchify, my_patchify)
drill(test_interpolate, my_interpolate_pos_embed)